In [0]:
use catalog claims;

In [0]:
create schema if not exists genai;

In [0]:
create table if not exists genai.discharge_extracted (
    claim_id string,
    source_file string,
    source string,
    document_text string,
    diagnosis_code string,
    procedure_code string,
    admission_date date,
    discharge_date date,
    room_type string
)

In [0]:
insert into genai.discharge_extracted
with parsed as (
    select element_at(split(path, '/'), -1) as source_file,
    element_at(split(path, '\\.'), -1) as source,
    ai_parse_document(content, map('version', '2.0')) as doc
    from read_files('/Volumes/claims/raw/landing/discharge_summaries', format => 'binaryFile')
),
extracted as (
    select source_file,
    source,
    doc,
    ai_extract (
        doc, 
        '{
        "claim_id" : {"type": "string", "description": "claim identifier referenced in the discharge summary"},
        "diagnosis_code" : {"type": "string", "description": "ICD-10 code of the final diagnosis for this admssion"},
        "procedure_code" : {"type": "string", "description": "Procedure code for the procedure performend duing this summary"},
        "admission_date" : {"type": "string", "description": "Date of admission, yyyy-MM-dd"},
        "discharge_date" : {"type" : "string", "description": "Date of discharge, yyyy-MM-dd"},
        "room_type" : {"type" : "enum", "labels": ["general_ward", "twin_sharing", "single_private", "deluxe"]}
        }',
        map('version', '2.1')
    ) as x 
    from parsed
)

SELECT
  x:response.claim_id.value::string AS claim_id,
  source_file,
  source,
  concat_ws('\n\n',
    transform(
      try_cast(doc:document:elements AS ARRAY<VARIANT>),
      element -> try_cast(element:content AS STRING)
    )
  ) AS document_text,
  x:response.diagnosis_code.value::string AS diagnosis_code,
  x:response.procedure_code.value::string AS procedure_code,
  try_cast(x:response.admission_date.value::string AS DATE) AS admission_date,
  try_cast(x:response.discharge_date.value::string AS DATE) AS discharge_date,
  x:response.room_type.value::string AS room_type
FROM extracted;

In [0]:
WITH compared AS (
  SELECT
    e.claim_id,
    e.diagnosis_code  = c.diagnosis_code  AS diagnosis_match,
    e.procedure_code  = c.procedure_code  AS procedure_match,
    e.admission_date  = c.admission_date  AS admission_match,
    e.discharge_date  = c.discharge_date  AS discharge_match
  FROM claims.genai.discharge_extracted e
  JOIN claims.silver.claims_clean c
    ON e.claim_id = c.claim_id
)

SELECT
  COUNT(*) AS claims_compared,

  -- per-field match rates, to see which field is dragging the number down
  ROUND(AVG(CASE WHEN diagnosis_match THEN 1.0 ELSE 0 END), 2) AS diagnosis_code_match_rate,
  ROUND(AVG(CASE WHEN procedure_match THEN 1.0 ELSE 0 END), 2) AS procedure_code_match_rate,
  ROUND(AVG(CASE WHEN admission_match THEN 1.0 ELSE 0 END), 2) AS admission_date_match_rate,
  ROUND(AVG(CASE WHEN discharge_match THEN 1.0 ELSE 0 END), 2) AS discharge_date_match_rate,

  -- overall: average of all 4 fields across all rows (this is the number that must be >= 0.90)
  ROUND(
    AVG(
      (CASE WHEN diagnosis_match THEN 1 ELSE 0 END
       + CASE WHEN procedure_match THEN 1 ELSE 0 END
       + CASE WHEN admission_match THEN 1 ELSE 0 END
       + CASE WHEN discharge_match THEN 1 ELSE 0 END) / 4.0
    ),
    2
  ) AS overall_match_rate

FROM compared;

In [0]:
create table if not exists claims.genai.grievance_emails
from read_files('/Volumes/claims/raw/landing/grievance_emails.csv', format => 'csv', header => true, inferSchema => true)

In [0]:
select * from genai.grievance_emails limit 5

In [0]:
CREATE OR REPLACE TABLE claims.genai.grievance_classified AS
SELECT
  email_id,
  ai_classify(
    CONCAT(COALESCE(subject, ''), '. ', COALESCE(body, '')),
    array('claim_delay', 'claim_rejection', 'payout_deduction', 'cashless_network', 'policy_service')
  ) AS category
FROM claims.genai.grievance_emails